# Meta Policy Strategy Training

This notebook trains the top-level meta policy whose action is the selected strategy.

It chooses among the three pretrained low-level strategy policies:

- `lr_calculation`
- `lr_heuristic`
- `dt_traversal`

The meta-policy always has the same action space, `Discrete(3)`, but actions are anonymous slots rather than fixed strategy IDs. At the start of each episode, the three low-level strategies are randomly assigned to slots `0..2`; this assignment stays fixed within the episode. In `mixed` mode, each episode randomly samples one of `linear_regression`, `decision_tree`, or `hybrid`, but knowing the condition alone no longer reveals which slot corresponds to a preferred strategy.

New models use `MaskablePPO`: slots whose underlying strategy is unavailable in the episode condition are masked out. In the LR condition, the policy sees two legal anonymous slots without learning which is calculation or heuristic; in the DT-only condition there is necessarily just one legal action.

## Conditions

- `linear_regression`: the anonymous slots containing `lr_calculation` and `lr_heuristic` are legal; the DT slot is masked.
- `decision_tree`: only the anonymous slot containing `dt_traversal` is legal.
- `hybrid`: all three strategies are available. On each trial, either no explanation, an LR explanation, or a DT explanation is shown. If the chosen strategy's own explanation is not shown, its low-level policy cannot read that explanation and must behave as a retrieval/memory-based strategy.

The hybrid condition fixes complexity per episode: low complexity uses sparse LR and depth-2 DT explanations; high complexity uses dense LR and depth-3 DT explanations.

## Observation Space

The combined observation has 43 dimensions in the default configuration (`history_window=5`):

1. mean probability-correct for each anonymous strategy slot over the previous five occurrences of each explanation type (`none`, `lr`, `dt`): `9`
2. mean prediction-time for each anonymous strategy slot over the previous five occurrences of each explanation type (`none`, `lr`, `dt`): `9`
3. previous five selected anonymous slots, each encoded as a three-way one-hot slot: `15`
4. normalized cognitive parameters: decision noise, memory retrieval threshold, opportunity cost: `3`
5. episode condition one-hot: `linear_regression`, `decision_tree`, `hybrid`: `3`
6. current explanation one-hot: `none`, `lr`, `dt`: `3`
7. episode progress: `1`

The history summaries are arranged slot-major: the three explanation buckets for slot `0`, followed by slot `1`, followed by slot `2`. Strategy-to-slot identity is deliberately absent from the observation. `selected_strategy` and `strategy_slot_order` are emitted only in evaluation `info` for analysis.

Older saved meta-policy models cannot be reused here: fixed-slot PPO checkpoints reveal strategy identity, and earlier masked-slot checkpoints do not have the explanation-conditioned history dimensions.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "rl_agents":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
from rl_agents.meta_policy_strategy import (
    CONDITIONS,
    META_STRATEGIES,
    CombinedPolicyConfig,
    combined_observation_action_summary,
    default_combined_parameter_sweep_values,
    evaluate_combined_policy,
    load_combined_bundles,
    load_combined_policy,
    plot_combined_parameter_sweep,
    plot_meta_strategy_selection_by_dataset,
    summarize_combined_evaluation,
    summarize_combined_parameter_sweep,
    summarize_meta_strategy_selection_by_dataset,
    sweep_combined_parameters,
    train_combined_policy,
)


In [ ]:
# Point this at the output folder produced by individual_policy_training_f.ipynb.
LOW_LEVEL_RUN = ROOT / "outputs" / "unified_strategy_policy" / "unified_strategy_demo"

combined_config = CombinedPolicyConfig(
    data_dir=str(ROOT / "datasets"),
    output_root=str(ROOT / "outputs" / "combined_strategy_meta_policy"),
    run_name="meta_policy_strategy_masked_explanation_history_demo",
    condition_name="mixed",
    total_timesteps=6e5,
    n_envs=4,
    instances_per_episode=40,
    max_features=6,
    explanation_shown_ratio=0.5,
    history_window=5,
    simulation_sample_count=16,
    memory_recall_threshold_min=-5.0,
    unavailable_strategy_penalty=-1.0,
    randomize_strategy_order_per_episode=True,
    meta_history_by_explanation=True,
    apps=None,
    lr_calculation_model_path=str(LOW_LEVEL_RUN / "lr_calculation" / "models" / "final_model.zip"),
    lr_heuristic_model_path=str(LOW_LEVEL_RUN / "lr_heuristic" / "models" / "final_model.zip"),
    dt_traversal_model_path=str(LOW_LEVEL_RUN / "dt_traversal" / "models" / "final_model.zip"),
)

asdict(combined_config)

In [ ]:
combined_observation_action_summary(combined_config)

In [ ]:
bundles = load_combined_bundles(ROOT / "datasets", combined_config)
pd.DataFrame([
    {
        "app_id": bundle.app_id,
        "model_name": bundle.model_name,
        "instances": len(bundle.instance_ids),
    }
    for bundle in bundles
])

In [ ]:
for strategy_name, model_path in {
    "lr_calculation": combined_config.lr_calculation_model_path,
    "lr_heuristic": combined_config.lr_heuristic_model_path,
    "dt_traversal": combined_config.dt_traversal_model_path,
}.items():
    path = Path(model_path)
    print(f"{strategy_name}: {path} exists={path.exists()}")

## Train Mixed-Condition Model

This trains one shared masked meta-policy. Each episode randomly uses `linear_regression`, `decision_tree`, or `hybrid`; it also reshuffles the anonymous strategy slots. Correctness and timing summaries keep the last five observations separately for each shown explanation type and remain arranged by anonymous slot, so selecting action slot `i` can be learned from the history features for slot `i` without revealing its strategy identity. Train a fresh model because earlier checkpoints have a different observation contract.

In [ ]:
# Keep this False after adding explanation-conditioned slot histories; older models are incompatible.
REUSE_SAVED_MODEL = False
COMBINED_MODEL_PATH = None  # Optional explicit path to final_model.zip

if REUSE_SAVED_MODEL:
    model, run_dir, trained_bundles = load_combined_policy(
        combined_config,
        model_path=COMBINED_MODEL_PATH,
    )
else:
    model, run_dir, trained_bundles = train_combined_policy(combined_config)

run_dir


## Optional Single-Condition Training

Use this only when you intentionally want a condition-specific baseline instead of the shared mixed-condition model.

In [ ]:
single_condition_config = CombinedPolicyConfig(
    **{
        **asdict(combined_config),
        "condition_name": "hybrid",
        "run_name": "combined_strategy_meta_hybrid_only",
    }
)

# model, run_dir, trained_bundles = train_combined_policy(single_condition_config)
# run_dir

## Evaluate

The evaluation output includes the anonymous selected slot, its diagnostic `strategy_slot_order` mapping, the actual selected strategy, the explanation type shown, and the selected probability/time. The mapping is for analysis only; it is not part of the policy observation.

In [ ]:
evaluation_df = evaluate_combined_policy(
    model,
    trained_bundles,
    combined_config,
    n_episodes=30,
    deterministic=True,
)
eval_path = run_dir / "metrics" / "evaluation.csv"
evaluation_df.to_csv(eval_path, index=False)
if not evaluation_df.empty:
    for (app_id, model_name), dataset_eval_df in evaluation_df.groupby(["app_id", "model_name"], dropna=False):
        print(f"\nSample rows: {app_id} / {model_name}")
        display(dataset_eval_df.head())


In [ ]:
evaluation_summary = summarize_combined_evaluation(evaluation_df)
if not evaluation_summary.empty:
    for (app_id, model_name), dataset_summary in evaluation_summary.groupby(["app_id", "model_name"], dropna=False):
        print(f"\n=== {app_id} / {model_name} ===")
        display(dataset_summary.drop(columns=["app_id", "model_name"]))


## Cognitive Parameter Sweep

This evaluates the trained mixed-condition meta-policy while clamping one cognitive parameter at a time. The output shows how often each strategy is selected as `decision_noise`, `memory_recall_threshold`, or `opportunity_cost` changes.


In [ ]:
sweep_values = default_combined_parameter_sweep_values(combined_config, n_points=5)
sweep_values


In [ ]:
combined_sweep_df = sweep_combined_parameters(
    model,
    trained_bundles,
    combined_config,
    sweep_values=sweep_values,
    n_episodes=20,
    deterministic=True,
)
combined_sweep_path = run_dir / "metrics" / "cognitive_parameter_sweep.csv"
combined_sweep_df.to_csv(combined_sweep_path, index=False)
combined_sweep_df.head()


In [ ]:
selection_summary = summarize_combined_parameter_sweep(combined_sweep_df)
display(
    selection_summary.sort_values(["sweep_parameter", "condition_name", "sweep_value", "selected_strategy"])
)


In [ ]:
sweep_figures = plot_combined_parameter_sweep(
    combined_sweep_df,
    output_dir=run_dir / "metrics" / "cognitive_parameter_sweep_plots",
    show=True,
)

dataset_selection_summary = summarize_meta_strategy_selection_by_dataset(combined_sweep_df)
dataset_selection_summary.to_csv(
    run_dir / "metrics" / "meta_strategy_selection_by_dataset_opportunity_cost.csv",
    index=False,
)
dataset_selection_figures = plot_meta_strategy_selection_by_dataset(
    combined_sweep_df,
    output_dir=run_dir / "metrics" / "cognitive_parameter_sweep_plots",
    show=True,
)

display(dataset_selection_summary.head(30))
list(sweep_figures.keys()) + list(dataset_selection_figures.keys())
